# Which combinations lead to a thyroid tumor or goitre? (parallel sets prototype)

Kaggle patients flow left to right through sex → age → pregnancy → diagnosis → outcome. Use the buttons to switch the outcome between **tumor** and **goitre**. Ribbon height is the number of patients. Orange ribbons are patients with the outcome; gray ribbons are everyone else. Each category is labeled with its size and its outcome rate.

- `tumor` and `goitre` are yes/no answers on the referral form, not confirmed diagnoses.
- Diagnosis group comes from the first letter of `target`: A–D hyper, E–H hypo, `-` normal, everything else "other".
- These are associations, not causes. Pregnant patients are all women and mostly young, for example.
- Goitre is rare (under 1%), so its combinations rest on a handful of patients.
- Rows with an impossible age (over 100) or no sex are dropped.
- The buttons need a running kernel. Run all cells, then click.

In [1]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.path import Path as MplPath

DATA = Path("..") / "data"
GROUP_BY_CODE = {**{c: "hyper" for c in "ABCD"}, **{c: "hypo" for c in "EFGH"}, "-": "normal"}
OUTCOMES = ["tumor", "goitre"]

df = pd.read_csv(DATA / "thyroidDF.csv", usecols=["age", "sex", "pregnant", "target", *OUTCOMES])
df = df[(df["age"] <= 100) & df["sex"].notna()].copy()
df["sex"] = df["sex"].map({"F": "female", "M": "male"})
df["age_band"] = pd.cut(df["age"], [0, 30, 45, 60, 100], labels=["under 30", "30–45", "45–60", "60+"]).astype(str)
df["pregnant"] = df["pregnant"].map({"t": "pregnant", "f": "not pregnant"})
df["diagnosis"] = df["target"].astype(str).str[0].map(GROUP_BY_CODE).fillna("other")
for o in OUTCOMES:
    df[o] = df[o].map({"t": o, "f": f"no {o}"})

FACTORS = {
    "sex": ("Sex", ["female", "male"]),
    "age_band": ("Age", ["under 30", "30–45", "45–60", "60+"]),
    "pregnant": ("Pregnancy", ["pregnant", "not pregnant"]),
    "diagnosis": ("Diagnosis", ["hyper", "normal", "hypo", "other"]),
}


def dims_for(outcome):
    return {**FACTORS, outcome: (outcome.capitalize(), [outcome, f"no {outcome}"])}


HIGHLIGHT, OTHER = "#eb6834", "#a3a29c"
SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#a3a29c"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "text.color": INK, "font.size": 10})

for o in OUTCOMES:
    print(f"{o}: {(df[o] == o).sum()} of {len(df):,} patients ({100 * (df[o] == o).mean():.1f}%)")

tumor: 236 of 8,861 patients (2.7%)
goitre: 84 of 8,861 patients (0.9%)


## Parallel sets

**Top:** all patients, with outcome patients in orange. Each ribbon keeps its outcome patients along its top edge.
**Bottom:** outcome patients only, so their paths are easier to trace. Labels there show each category's share of those patients.
**Table:** every sex → age → pregnancy → diagnosis path with at least 30 patients, ranked by outcome rate.

In [2]:
NODE_W = 0.035


def ribbon(ax, x0, s_top, s_bot, x1, t_top, t_bot, color, alpha, edge):
    xm = (x0 + x1) / 2
    verts = [(x0, s_top), (xm, s_top), (xm, t_top), (x1, t_top),
             (x1, t_bot), (xm, t_bot), (xm, s_bot), (x0, s_bot), (x0, s_top)]
    codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
             MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CLOSEPOLY]
    ax.add_patch(PathPatch(MplPath(verts, codes), facecolor=color, alpha=alpha, edgecolor=SURFACE, lw=0.6 if edge else 0))


def parallel_sets(ax, data, dims, outcome, label_fn, edge=False):
    cols = list(dims)
    gap = 0.025 * len(data)
    has = data[outcome] == outcome
    order = [outcome, f"no {outcome}"]

    # Node extents: categories stacked top to bottom, sized by patient count.
    nodes = {}
    for col in cols:
        y = 0.0
        for cat in dims[col][1]:
            mask = data[col] == cat
            nodes[(col, cat)] = {"top": y, "bottom": y + mask.sum(), "n": int(mask.sum()),
                                 "rate": has[mask].mean() if mask.any() else 0}
            y += mask.sum() + gap

    for i, (a, b) in enumerate(zip(cols, cols[1:])):
        flows = data.groupby([a, b, outcome]).size()
        cats_a, cats_b = dims[a][1], dims[b][1]
        # Ribbons ordered by the other column's category, so each ends at its real node;
        # within a ribbon, outcome patients sit on its top edge.
        src_pos, dst_pos = {}, {}
        for ca in cats_a:
            off = nodes[(a, ca)]["top"]
            for cb in cats_b:
                for t in order:
                    n = flows.get((ca, cb, t), 0)
                    src_pos[(ca, cb, t)] = (off, off + n)
                    off += n
        for cb in cats_b:
            off = nodes[(b, cb)]["top"]
            for ca in cats_a:
                for t in order:
                    n = flows.get((ca, cb, t), 0)
                    dst_pos[(ca, cb, t)] = (off, off + n)
                    off += n
        for layer in reversed(order):  # orange drawn last, on top
            for (ca, cb, t), n in flows.items():
                if t != layer or n == 0:
                    continue
                (s0, s1), (d0, d1) = src_pos[(ca, cb, t)], dst_pos[(ca, cb, t)]
                ribbon(ax, i + NODE_W / 2, s0, s1, i + 1 - NODE_W / 2, d0, d1,
                       HIGHLIGHT if t == outcome else OTHER, 0.9 if t == outcome else 0.28, edge)

    for i, col in enumerate(cols):
        title, cats = dims[col]
        ax.text(i, -0.05 * len(data), title, ha="center", va="bottom", fontsize=11, fontweight="bold")
        for cat in cats:
            nd = nodes[(col, cat)]
            if nd["n"] == 0:
                continue
            ax.add_patch(Rectangle((i - NODE_W / 2, nd["top"]), NODE_W, nd["n"], color=INK, lw=0))
            ax.text(i + NODE_W, (nd["top"] + nd["bottom"]) / 2, label_fn(col, cat, nd), ha="left",
                    va="center", fontsize=8, color=INK,
                    bbox=dict(boxstyle="round,pad=0.2", facecolor=SURFACE, edgecolor="none", alpha=0.85))

    last = max(nd["bottom"] for nd in nodes.values())
    ax.set_xlim(-0.1, len(cols) - 0.45)
    ax.set_ylim(last + gap, -0.08 * len(data))
    ax.axis("off")


def draw(outcome):
    dims = dims_for(outcome)
    only = df[df[outcome] == outcome]

    def label_all(col, cat, nd):
        if col == outcome:
            return f"{cat}\nn = {nd['n']:,}"
        return f"{cat}\nn = {nd['n']:,} · {100 * nd['rate']:.1f}% {outcome}"

    def label_only(col, cat, nd):
        return f"{cat}\n{nd['n']} · {100 * nd['n'] / len(only):.0f}% of {outcome} patients"

    fig, (ax_all, ax_o) = plt.subplots(2, 1, figsize=(13, 13), gridspec_kw={"height_ratios": [1.1, 1], "hspace": 0.12})
    parallel_sets(ax_all, df, dims, outcome, label_all)
    parallel_sets(ax_o, only, FACTORS, outcome, label_only, edge=True)
    ax_all.set_title(f"All {len(df):,} patients", loc="left", fontsize=11, color=INK2, pad=22)
    ax_o.set_title(f"Only the {len(only)} patients with a {outcome}", loc="left", fontsize=11, color=INK2, pad=22)
    fig.suptitle(f"Which combinations lead to a thyroid {outcome}?", x=0.125, y=0.975, ha="left", fontsize=14)
    fig.text(0.125, 0.952, f"Orange = patients with a {outcome} ({100 * len(only) / len(df):.1f}% overall).",
             fontsize=9, color=INK2)
    fig.savefig(f"parallel_sets_{outcome}.png", dpi=110, bbox_inches="tight")
    plt.show()


def path_table(outcome, min_patients=30):
    paths = (df.groupby(["sex", "age_band", "pregnant", "diagnosis"])
               .agg(patients=(outcome, "size"), cases=(outcome, lambda s: (s == outcome).sum())))
    paths[f"{outcome}_rate_%"] = (100 * paths["cases"] / paths["patients"]).round(1)
    return paths[paths["patients"] >= min_patients].sort_values(f"{outcome}_rate_%", ascending=False).head(12)

In [3]:
buttons = widgets.ToggleButtons(options=[("Tumor", "tumor"), ("Goitre", "goitre")], value="tumor",
                                description="Outcome:")
out = widgets.Output()


def show(outcome):
    with out:
        out.clear_output(wait=True)
        draw(outcome)
        display(path_table(outcome))


buttons.observe(lambda change: show(change["new"]), names="value")
display(buttons, out)
show(buttons.value)

ToggleButtons(description='Outcome:', options=(('Tumor', 'tumor'), ('Goitre', 'goitre')), value='tumor')

Output()